# 02 - Tratamento Prata: data_api

Lê a camada **Bronze** (`lakehouse.bronze.data_api`), aplica limpeza,
padronização de tipos/strings e deduplicação, e grava na camada **Prata**
em formato Iceberg, no bucket `lakehouse/prata/data_api`.

Regras aplicadas:
- Remoção de duplicados por `id` (mantendo o registro com `data_carga` mais recente)
- Padronização de strings: `category` em minúsculas e sem espaços extras,
  `title`/`description` com espaços normalizados
- Tratamento de nulos: `price`, `rating_rate`, `rating_count` -> 0 quando nulos
- Tipagem explícita e arredondamento de `price` (2 casas decimais)
- Coluna técnica `data_processamento` registrando quando o tratamento ocorreu

In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (
    col, lower, trim, regexp_replace, coalesce, lit, round as spark_round,
    current_timestamp, row_number
)

## 1. Configuração da SparkSession

In [ ]:
spark = (
    SparkSession.builder
    .appName("prata-data-api")
    .config("spark.jars.packages",
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1,"
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
            "org.postgresql:postgresql:42.7.4")

    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "jdbc")
    .config("spark.sql.catalog.lakehouse.uri", "jdbc:postgresql://postgres:5432/metastore")
    .config("spark.sql.catalog.lakehouse.jdbc.user", "dba_admin")
    .config("spark.sql.catalog.lakehouse.jdbc.password", "Agent9-Backtalk6-Zestfully5-Luxury1-Calculus2")
    .config("spark.sql.catalog.lakehouse.jdbc.driver", "org.postgresql.Driver")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3a://lakehouse/warehouse")

    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin_minio")
    .config("spark.hadoop.fs.s3a.secret.key", "Grunt9-Relenting6-Hula5-Catcall9-Residue3")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")

    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

## 2. Leitura da camada Bronze

In [ ]:
df_bronze = spark.table("lakehouse.bronze.data_api")
df_bronze.printSchema()
print(f"Total de registros na Bronze: {df_bronze.count()}")

## 3. Deduplicação

Mantém apenas o registro mais recente por `id` (baseado em `data_carga`).

In [ ]:
janela = Window.partitionBy("id").orderBy(col("data_carga").desc())

df_dedup = (
    df_bronze
    .withColumn("_rn", row_number().over(janela))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

print(f"Total de registros após deduplicação: {df_dedup.count()}")

## 4. Padronização de tipos e strings

In [ ]:
df_prata = (
    df_dedup
    .withColumn("title", trim(regexp_replace(col("title"), r"\s+", " ")))
    .withColumn("description", trim(regexp_replace(col("description"), r"\s+", " ")))
    .withColumn("category", lower(trim(col("category"))))
    .withColumn("price", spark_round(coalesce(col("price"), lit(0.0)), 2))
    .withColumn("rating_rate", coalesce(col("rating_rate"), lit(0.0)))
    .withColumn("rating_count", coalesce(col("rating_count"), lit(0)))
    .withColumn("data_processamento", current_timestamp())
    .select(
        "id", "title", "price", "description", "category", "image",
        "rating_rate", "rating_count", "data_carga", "data_processamento"
    )
)

df_prata.show(5, truncate=40)

## 5. Validação básica de qualidade

Garante que não restaram nulos em colunas-chave e que `price` não é negativo.

In [ ]:
qtd_id_nulo = df_prata.filter(col("id").isNull()).count()
qtd_price_negativo = df_prata.filter(col("price") < 0).count()

print(f"Registros com id nulo: {qtd_id_nulo}")
print(f"Registros com price negativo: {qtd_price_negativo}")

assert qtd_id_nulo == 0, "Existem registros com id nulo na camada Prata!"
assert qtd_price_negativo == 0, "Existem registros com price negativo na camada Prata!"

## 6. Criar schema e tabela Iceberg na camada Prata

Localização: `s3a://lakehouse/prata/data_api`

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS lakehouse.prata")

spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.prata.data_api (
    id                 INT,
    title              STRING,
    price              DOUBLE,
    description        STRING,
    category           STRING,
    image              STRING,
    rating_rate        DOUBLE,
    rating_count       INT,
    data_carga         TIMESTAMP,
    data_processamento TIMESTAMP
)
USING iceberg
LOCATION 's3a://lakehouse/prata/data_api'
""")

## 7. Carga incremental (MERGE / upsert por `id`)

In [ ]:
df_prata.createOrReplaceTempView("staging_prata")

spark.sql("""
MERGE INTO lakehouse.prata.data_api AS target
USING staging_prata AS source
ON target.id = source.id
WHEN MATCHED THEN UPDATE SET
    target.title              = source.title,
    target.price              = source.price,
    target.description        = source.description,
    target.category           = source.category,
    target.image              = source.image,
    target.rating_rate        = source.rating_rate,
    target.rating_count       = source.rating_count,
    target.data_carga         = source.data_carga,
    target.data_processamento = source.data_processamento
WHEN NOT MATCHED THEN INSERT (
    id, title, price, description, category, image,
    rating_rate, rating_count, data_carga, data_processamento
)
VALUES (
    source.id, source.title, source.price, source.description, source.category,
    source.image, source.rating_rate, source.rating_count,
    source.data_carga, source.data_processamento
)
""")

## 8. Verificação

In [ ]:
spark.sql("SELECT COUNT(*) AS total_registros FROM lakehouse.prata.data_api").show()
spark.sql("SELECT category, COUNT(*) AS qtd FROM lakehouse.prata.data_api GROUP BY category ORDER BY qtd DESC").show()
spark.sql("SELECT * FROM lakehouse.prata.data_api ORDER BY id LIMIT 10").show(truncate=30)

In [ ]:
spark.stop()